In [ ]:
!pip install openai
!pip install azure-search-documents==11.4.0

In [ ]:
import openai
# from azure.storage.blob import BlobServiceClient
import pickle
# import numpy as np
import os

from azure.search.documents.indexes.models import (  
    SearchIndex,  
    SearchField,  
    SearchFieldDataType,  
    SimpleField,  
    SearchableField,  
    VectorSearch,
    VectorSearchProfile,
    HnswAlgorithmConfiguration
)


In [ ]:
from azure.search.documents.indexes import SearchIndexClient
from azure.search.documents.indexes.models import SearchIndex, SimpleField, SearchFieldDataType, SearchableField
from azure.core.credentials import AzureKeyCredential
from azure.search.documents import SearchClient
import re

In [ ]:
# query_documents = [
#     {
#         "id": "query_001",
#         "query_desc": "Get all employees with their department names",
#         "sql_query": """
#             SELECT e.emp_id, e.emp_name, e.emp_salary, d.dept_name
#             FROM employee e
#             LEFT JOIN department d ON e.dept_id = d.dept_id;
#         """
#     },
#     {
#         "id": "query_002",
#         "query_desc": "Find the highest-paid employee",
#         "sql_query": """
#             SELECT emp_name, emp_salary 
#             FROM employee 
#             ORDER BY emp_salary DESC 
#             LIMIT 1;
#         """
#     },
#     {
#         "id": "query_003",
#         "query_desc": "Count the number of employees in each department",
#         "sql_query": """
#             SELECT d.dept_name, COUNT(e.emp_id) as employee_count
#             FROM department d
#             LEFT JOIN employee e ON d.dept_id = e.dept_id
#             GROUP BY d.dept_name;
#         """
#     },
#     {
#         "id": "query_004",
#         "query_desc": "Get total salary expense for each department",
#         "sql_query": """
#             SELECT d.dept_name, SUM(e.emp_salary) as total_salary
#             FROM department d
#             LEFT JOIN employee e ON d.dept_id = e.dept_id
#             GROUP BY d.dept_name;
#         """
#     },
#     {
#         "id": "query_005",
#         "query_desc": "Find employees who earn more than 70,000",
#         "sql_query": """
#             SELECT emp_id, emp_name, emp_salary
#             FROM employee
#             WHERE emp_salary > 70000;
#         """
#     },
#     {
#         "id": "query_006",
#         "query_desc": "Get employees who do not belong to any department",
#         "sql_query": """
#             SELECT emp_id, emp_name
#             FROM employee
#             WHERE dept_id IS NULL;
#         """
#     },
#     {
#         "id": "query_007",
#         "query_desc": "Find the average salary per department",
#         "sql_query": """
#             SELECT d.dept_name, AVG(e.emp_salary) as avg_salary
#             FROM department d
#             LEFT JOIN employee e ON d.dept_id = e.dept_id
#             GROUP BY d.dept_name;
#         """
#     },
#     {
#         "id": "query_008",
#         "query_desc": "Retrieve employees whose name starts with 'J'",
#         "sql_query": """
#             SELECT emp_id, emp_name
#             FROM employee
#             WHERE emp_name LIKE 'J%';
#         """
#     }
# ]

query_documents = [
    {
        "id": "query_001",
        "query_desc": "Get all employees with their department names",
        "sql_query": """
            SELECT e.emp_id, e.emp_name, e.emp_salary, d.dept_name
            FROM employee e
            LEFT JOIN department d ON e.dept_id = d.dept_id;
        """
    },
    {
        "id": "query_002",
        "query_desc": "Find the highest-paid employee",
        "sql_query": """
            SELECT emp_name, emp_salary 
            FROM employee 
            ORDER BY emp_salary DESC 
            LIMIT 1;
        """
    },
    {
        "id": "query_003",
        "query_desc": "Count the number of employees in each department",
        "sql_query": """
            SELECT d.dept_name, COUNT(e.emp_id) as employee_count
            FROM department d
            LEFT JOIN employee e ON d.dept_id = e.dept_id
            GROUP BY d.dept_name;
        """
    },
    {
        "id": "query_004",
        "query_desc": "Get total salary expense for each department",
        "sql_query": """
            SELECT d.dept_name, SUM(e.emp_salary) as total_salary
            FROM department d
            LEFT JOIN employee e ON d.dept_id = e.dept_id
            GROUP BY d.dept_name;
        """
    },
    {
        "id": "query_005",
        "query_desc": "Find employees who earn more than 70,000",
        "sql_query": """
            SELECT emp_id, emp_name, emp_salary
            FROM employee
            WHERE emp_salary > 70000;
        """
    },
    {
        "id": "query_006",
        "query_desc": "Get employees who do not belong to any department",
        "sql_query": """
            SELECT emp_id, emp_name
            FROM employee
            WHERE dept_id IS NULL;
        """
    },
    {
        "id": "query_007",
        "query_desc": "Find the average salary per department",
        "sql_query": """
            SELECT d.dept_name, AVG(e.emp_salary) as avg_salary
            FROM department d
            LEFT JOIN employee e ON d.dept_id = e.dept_id
            GROUP BY d.dept_name;
        """
    },
    {
        "id": "query_008",
        "query_desc": "Retrieve employees whose name starts with 'J'",
        "sql_query": """
            SELECT emp_id, emp_name
            FROM employee
            WHERE emp_name LIKE 'J%';
        """
    },
    {
        "id": "query_009",
        "query_desc": "Get employees hired in the last year",
        "sql_query": """
            SELECT emp_id, emp_name, hire_date
            FROM employee
            WHERE hire_date >= DATE_SUB(CURDATE(), INTERVAL 1 YEAR);
        """
    },
    {
        "id": "query_010",
        "query_desc": "Find employees with the same salary",
        "sql_query": """
            SELECT emp_salary, COUNT(*) as count
            FROM employee
            GROUP BY emp_salary
            HAVING COUNT(*) > 1;
        """
    },
    {
        "id": "query_011",
        "query_desc": "Find department with the most employees",
        "sql_query": """
            SELECT d.dept_name, COUNT(e.emp_id) as employee_count
            FROM department d
            LEFT JOIN employee e ON d.dept_id = e.dept_id
            GROUP BY d.dept_name
            ORDER BY employee_count DESC
            LIMIT 1;
        """
    },
    {
        "id": "query_012",
        "query_desc": "Find employees with missing email addresses",
        "sql_query": """
            SELECT emp_id, emp_name
            FROM employee
            WHERE email IS NULL OR email = '';
        """
    },
    {
        "id": "query_013",
        "query_desc": "Get total number of employees",
        "sql_query": """
            SELECT COUNT(*) as total_employees
            FROM employee;
        """
    },
    {
        "id": "query_014",
        "query_desc": "Find employees who joined before 2015",
        "sql_query": """
            SELECT emp_id, emp_name, hire_date
            FROM employee
            WHERE hire_date < '2015-01-01';
        """
    },
    {
        "id": "query_015",
        "query_desc": "Get department-wise minimum salary",
        "sql_query": """
            SELECT d.dept_name, MIN(e.emp_salary) as min_salary
            FROM department d
            LEFT JOIN employee e ON d.dept_id = e.dept_id
            GROUP BY d.dept_name;
        """
    },
    {
        "id": "query_016",
        "query_desc": "Find top 3 highest-paying departments",
        "sql_query": """
            SELECT d.dept_name, SUM(e.emp_salary) as total_salary
            FROM department d
            LEFT JOIN employee e ON d.dept_id = e.dept_id
            GROUP BY d.dept_name
            ORDER BY total_salary DESC
            LIMIT 3;
        """
    },
    {
        "id": "query_017",
        "query_desc": "List employees along with their manager's name",
        "sql_query": """
            SELECT e.emp_name AS Employee, m.emp_name AS Manager
            FROM employee e
            LEFT JOIN employee m ON e.manager_id = m.emp_id;
        """
    },
    {
        "id": "query_018",
        "query_desc": "Find employees who have worked for more than 10 years",
        "sql_query": """
            SELECT emp_id, emp_name, hire_date
            FROM employee
            WHERE hire_date <= DATE_SUB(CURDATE(), INTERVAL 10 YEAR);
        """
    },
    {
        "id": "query_019",
        "query_desc": "Get employees who earn the department's average salary",
        "sql_query": """
            SELECT e.emp_id, e.emp_name, e.emp_salary, d.dept_name
            FROM employee e
            JOIN department d ON e.dept_id = d.dept_id
            WHERE e.emp_salary = (SELECT AVG(emp_salary) FROM employee WHERE dept_id = e.dept_id);
        """
    },
    {
        "id": "query_020",
        "query_desc": "Find employees who have not received a promotion",
        "sql_query": """
            SELECT emp_id, emp_name
            FROM employee
            WHERE last_promotion_date IS NULL;
        """
    },
    {
        "id": "query_021",
        "query_desc": "Get employees who have taken the most leave",
        "sql_query": """
            SELECT e.emp_name, COUNT(a.absent_date) as leave_days
            FROM employee e
            JOIN attendance a ON e.emp_id = a.emp_id
            WHERE a.status = 'Absent'
            GROUP BY e.emp_name
            ORDER BY leave_days DESC
            LIMIT 1;
        """
    },
    {
        "id": "query_022",
        "query_desc": "Get employees who have taken no leaves",
        "sql_query": """
            SELECT emp_id, emp_name
            FROM employee
            WHERE emp_id NOT IN (SELECT emp_id FROM attendance WHERE status = 'Absent');
        """
    },
    {
        "id": "query_023",
        "query_desc": "Find employees who have changed departments",
        "sql_query": """
            SELECT emp_id, emp_name, COUNT(DISTINCT dept_id) as dept_count
            FROM employee_history
            GROUP BY emp_id, emp_name
            HAVING dept_count > 1;
        """
    },
    {
        "id": "query_024",
        "query_desc": "Get employees and their performance ratings",
        "sql_query": """
            SELECT e.emp_name, p.rating
            FROM employee e
            JOIN performance p ON e.emp_id = p.emp_id;
        """
    },
    {
        "id": "query_025",
        "query_desc": "Find employees who have received a salary increase",
        "sql_query": """
            SELECT emp_id, emp_name
            FROM employee_salary_history
            WHERE salary > previous_salary;
        """
    },
    {
        "id": "query_026",
        "query_desc": "List departments with no employees",
        "sql_query": """
            SELECT dept_name
            FROM department
            WHERE dept_id NOT IN (SELECT DISTINCT dept_id FROM employee);
        """
    }
]


In [ ]:
# Azure Cognitive Search details
search_service_name = ""
admin_api_key = ""
query_index_name = "query_index" #define new name

# Initialize Search Index client
endpoint = f"https://{search_service_name}.search.windows.net"
credential = AzureKeyCredential(admin_api_key)
index_client = SearchIndexClient(endpoint=endpoint, credential=credential)

# Define the index schema
def create_search_index():
    try:
        fields = [
            SimpleField(name="id", type=SearchFieldDataType.String, key=True),
            SearchableField(name="query_desc", type=SearchFieldDataType.String),
            SimpleField(name="sql_query", type=SearchFieldDataType.String)
        ]
        
        index = SearchIndex(name=query_index_name, fields=fields)
        index_client.create_index(index)
        print("✅ Query Index Created!")
    except Exception as e:
        print(f" Error Query Index Created!: {e}")
        raise

create_search_index()


In [ ]:
query_client = SearchClient(
            endpoint=endpoint,
            index_name=query_index_name,
            credential=credential
        )

# Upload Updated Queries

query_client.upload_documents(query_documents)
print("✅ Updated Queries Uploaded!")


In [ ]:
# # 🔹 Define schema metadata with descriptions, tags & synonyms
# schema_documents = [
#     {
#         "id": "schema_employee",
#         "table_name": "employee",
#         "description": "Stores employee details such as name, salary, and department.",
#         "schema_metadata": """
#             Table: employee
#             Description: Stores employee details such as name, salary, and department.
#             Columns:
#             - emp_id (INT, PK) → Unique identifier for each employee.
#             - emp_name (VARCHAR) → Full name of the employee.
#             - emp_salary (DECIMAL) → Salary of the employee.
#             - dept_id (INT, FK to department) → Foreign key linking to department table.
#         """,
#         "tags": ["HR", "Payroll", "Employee Management"],
#         "synonyms": ["emp", "staff", "worker"]
#     },
#     {
#         "id": "schema_department",
#         "table_name": "department",
#         "description": "Stores department details like department names.",
#         "schema_metadata": """
#             Table: department
#             Description: Stores department details like department names.
#             Columns:
#             - dept_id (INT, PK) → Unique identifier for each department.
#             - dept_name (VARCHAR) → Name of the department.
#         """,
#         "tags": ["Organization", "HR"],
#         "synonyms": ["dept", "division"]
#     }
# ]

schema_documents = [
    {
        "id": "schema_employee",
        "table_name": "employee",
        "description": "Stores employee details such as name, salary, and department.",
        "schema_metadata": """
            Table: employee
            Description: Stores employee details such as name, salary, and department.
            Columns:
            - emp_id (INT, PK) → Unique identifier for each employee.
            - emp_name (VARCHAR) → Full name of the employee.
            - emp_salary (DECIMAL) → Salary of the employee.
            - dept_id (INT, FK to department) → Foreign key linking to department table.
        """,
        "tags": ["HR", "Payroll", "Employee Management"],
        "synonyms": ["emp", "staff", "worker"]
    },
    {
        "id": "schema_department",
        "table_name": "department",
        "description": "Stores department details like department names.",
        "schema_metadata": """
            Table: department
            Description: Stores department details like department names.
            Columns:
            - dept_id (INT, PK) → Unique identifier for each department.
            - dept_name (VARCHAR) → Name of the department.
        """,
        "tags": ["Organization", "HR"],
        "synonyms": ["dept", "division"]
    },
    {
        "id": "schema_salary",
        "table_name": "salary",
        "description": "Contains employee salary details.",
        "schema_metadata": """
            Table: salary
            Description: Contains employee salary details.
            Columns:
            - salary_id (INT, PK) → Unique identifier for salary record.
            - emp_id (INT, FK to employee) → Employee receiving the salary.
            - amount (DECIMAL) → Salary amount.
            - pay_date (DATE) → Date salary was paid.
        """,
        "tags": ["Payroll", "Finance"],
        "synonyms": ["pay", "wages", "compensation"]
    },
    {
        "id": "schema_project",
        "table_name": "project",
        "description": "Stores details about company projects.",
        "schema_metadata": """
            Table: project
            Description: Stores details about company projects.
            Columns:
            - proj_id (INT, PK) → Unique identifier for each project.
            - proj_name (VARCHAR) → Name of the project.
            - start_date (DATE) → Start date of the project.
            - end_date (DATE) → Expected end date of the project.
        """,
        "tags": ["Project Management"],
        "synonyms": ["task", "initiative", "assignment"]
    },
    {
        "id": "schema_employee_project",
        "table_name": "employee_project",
        "description": "Links employees to projects they are assigned to.",
        "schema_metadata": """
            Table: employee_project
            Description: Links employees to projects they are assigned to.
            Columns:
            - emp_id (INT, FK to employee) → Employee assigned to project.
            - proj_id (INT, FK to project) → Project assigned to employee.
            - role (VARCHAR) → Employee role in project.
        """,
        "tags": ["HR", "Project Management"],
        "synonyms": ["team_assignment", "staff_project"]
    },
    {
        "id": "schema_attendance",
        "table_name": "attendance",
        "description": "Tracks employee attendance records.",
        "schema_metadata": """
            Table: attendance
            Description: Tracks employee attendance records.
            Columns:
            - att_id (INT, PK) → Unique identifier for attendance record.
            - emp_id (INT, FK to employee) → Employee present or absent.
            - att_date (DATE) → Date of attendance.
            - status (VARCHAR) → Present, Absent, or Late.
        """,
        "tags": ["HR", "Time Management"],
        "synonyms": ["timesheet", "presence", "absence"]
    },
    {
        "id": "schema_leave",
        "table_name": "leave",
        "description": "Manages employee leave requests.",
        "schema_metadata": """
            Table: leave
            Description: Manages employee leave requests.
            Columns:
            - leave_id (INT, PK) → Unique identifier for leave request.
            - emp_id (INT, FK to employee) → Employee requesting leave.
            - leave_type (VARCHAR) → Type of leave (Sick, Vacation, etc.).
            - start_date (DATE) → Start date of leave.
            - end_date (DATE) → End date of leave.
            - status (VARCHAR) → Approved, Pending, or Rejected.
        """,
        "tags": ["HR", "Time Off"],
        "synonyms": ["absence", "vacation", "PTO"]
    },
    {
        "id": "schema_inventory",
        "table_name": "inventory",
        "description": "Stores details about company inventory.",
        "schema_metadata": """
            Table: inventory
            Description: Stores details about company inventory.
            Columns:
            - item_id (INT, PK) → Unique identifier for inventory item.
            - item_name (VARCHAR) → Name of the item.
            - quantity (INT) → Available stock quantity.
            - location (VARCHAR) → Storage location of item.
        """,
        "tags": ["Assets", "Stock"],
        "synonyms": ["stock", "supplies", "items"]
    },
    {
        "id": "schema_orders",
        "table_name": "orders",
        "description": "Stores order details for purchases.",
        "schema_metadata": """
            Table: orders
            Description: Stores order details for purchases.
            Columns:
            - order_id (INT, PK) → Unique identifier for each order.
            - customer_id (INT, FK to customers) → Customer who placed the order.
            - order_date (DATE) → Date of order.
            - total_amount (DECIMAL) → Total order value.
        """,
        "tags": ["Sales", "Finance"],
        "synonyms": ["purchase", "transaction", "sale"]
    },
    {
        "id": "schema_customers",
        "table_name": "customers",
        "description": "Stores customer details.",
        "schema_metadata": """
            Table: customers
            Description: Stores customer details.
            Columns:
            - customer_id (INT, PK) → Unique identifier for customer.
            - customer_name (VARCHAR) → Full name of customer.
            - email (VARCHAR) → Contact email.
            - phone (VARCHAR) → Contact phone number.
        """,
        "tags": ["Sales", "CRM"],
        "synonyms": ["client", "buyer", "consumer"]
    },
    {
        "id": "schema_transactions",
        "table_name": "transactions",
        "description": "Records all financial transactions.",
        "schema_metadata": """
            Table: transactions
            Description: Records all financial transactions.
            Columns:
            - transaction_id (INT, PK) → Unique identifier for transaction.
            - order_id (INT, FK to orders) → Order related to transaction.
            - amount (DECIMAL) → Transaction amount.
            - transaction_date (DATE) → Date of transaction.
        """,
        "tags": ["Finance", "Accounting"],
        "synonyms": ["payment", "billing", "purchase"]
    },
    {
        "id": "schema_suppliers",
        "table_name": "suppliers",
        "description": "Stores supplier details.",
        "schema_metadata": """
            Table: suppliers
            Description: Stores supplier details.
            Columns:
            - supplier_id (INT, PK) → Unique identifier for supplier.
            - supplier_name (VARCHAR) → Name of the supplier.
            - contact_person (VARCHAR) → Supplier contact person.
            - phone (VARCHAR) → Contact phone number.
        """,
        "tags": ["Procurement", "Supply Chain"],
        "synonyms": ["vendor", "provider", "distributor"]
    }
]


In [ ]:
# Azure Cognitive Search details
search_service_name = "ssbi-ai-search"
admin_api_key = ""
schema_index_name = "schema_index" #define new name

# Initialize Search Index client
endpoint = f"https://{search_service_name}.search.windows.net"
credential = AzureKeyCredential(admin_api_key)
index_client = SearchIndexClient(endpoint=endpoint, credential=credential)

# Define the index schema
def create_search_index():
    try:
        fields = [
            SimpleField(name="id", type=SearchFieldDataType.String, key=True),
            SearchableField(name="table_name", type=SearchFieldDataType.String, analyzer_name="standard.lucene"),
            SearchableField(name="description", type=SearchFieldDataType.String, analyzer_name="standard.lucene"),
            SearchableField(name="schema_metadata", type=SearchFieldDataType.String, analyzer_name="standard.lucene"),
            SearchField(name="tags", type=SearchFieldDataType.Collection(SearchFieldDataType.String), searchable=True, filterable=True),
            SearchField(name="synonyms", type=SearchFieldDataType.Collection(SearchFieldDataType.String), searchable=True),
            SearchField(name="content_vector", type=SearchFieldDataType.Collection(SearchFieldDataType.Single), searchable=True, vector_search_dimensions=1536, vector_search_profile_name="my-vector-config")
        ]

        vector_search = VectorSearch(
            profiles=[VectorSearchProfile(name="my-vector-config", algorithm_configuration_name="my-algorithms-config")],
            algorithms=[HnswAlgorithmConfiguration(name="my-algorithms-config")],)
        
        index = SearchIndex(name=schema_index_name, fields=fields, vector_search=vector_search)
        index_client.create_index(index)
        print("✅ Query Index Created!")
    except Exception as e:
        print(f" Error Query Index Created!: {e}")
        raise

create_search_index()


In [ ]:
import openai


# Azure OpenAI Service client
client = openai.AzureOpenAI(
        azure_endpoint="<>",
        api_key="",
        api_version="2025-01-01-preview"
    )

model = "text-embedding-ada-002"

# # Generate embeddings using Azure OpenAI Service
# def generate_embeddings(texts):
#     try:
#         response = client.embeddings.create(input=texts, model=model).data
#         embeddings = [item.embedding for item in response]
        
#         # Print the number of embeddings and their shapes
#         print(f"Number of embeddings: {len(embeddings)}")
#         for i, embedding in enumerate(embeddings):
#             print(f"Embedding {i} shape: {len(embedding)}")
        
#         return embeddings
    
#     except Exception as e:
#         print(f"Error generating embeddings: {e}")
#         raise

# 🔹 Generate Embeddings using Azure OpenAI
def generate_embeddings(texts):
    try:
        response = client.embeddings.create(input=texts, model=model).data
        embeddings = [item.embedding for item in response]
        
        print(f"✅ Generated {len(embeddings)} embeddings")
        return embeddings
    
    except Exception as e:
        print(f"❌ Error generating embeddings: {e}")
        raise

In [ ]:
# 🔹 Generate embeddings and update schema documents
text_inputs = [
    f"{doc['table_name']} {doc['description']} {doc['schema_metadata']} {' '.join(doc['tags'])} {' '.join(doc['synonyms'])}"
    for doc in schema_documents
]

embeddings = generate_embeddings(text_inputs)

for i, doc in enumerate(schema_documents):
    doc["content_vector"] = embeddings[i]

In [ ]:
# schema_documents[1]

In [ ]:
print("Sample Embedding:", schema_documents[0]["content_vector"][:5])  # Print first 5 values


In [ ]:
schema_client = SearchClient(
            endpoint=endpoint,
            index_name=schema_index_name,
            credential=credential
        )

# Upload Updated Queries




In [ ]:
schema_client.upload_documents(documents=schema_documents)
print("✅ Updated Queries Uploaded!")

In [ ]:
user_query = "get the count of employees in each department"

In [ ]:

index_name = "schema_index"
def schema_search(query_text, top_k=5):
    try:
        search_client = SearchClient(
            endpoint=endpoint,
            index_name=index_name,
            credential=credential
        )

        from azure.search.documents.models import VectorizedQuery

        # Generate embedding for the query
        query_embedding = generate_embeddings([query_text])[0]  # Ensure it's a single embedding

        vector_query = VectorizedQuery(vector=query_embedding, k_nearest_neighbors=5, fields="content_vector")

        results = search_client.search(search_text="", vector_queries=[vector_query], top=top_k, 
                                        #query_type="semantic",  # Enables AI-driven ranking (optional)
                                        #select=["table_name", "schema_metadata", "@search.score"],  # Select fields
                                        #semantic_configuration_name="my-semantic-config"  # Enable if using semantic search
            )
        
        # print(results)

       

        # Print results
        for result in results:
            print(f"✅ Found: {result['table_name']}")
            print(f"🔹 Schema Metadata: {result['schema_metadata']}")
            print(f"⭐ Score: {result['@search.score']}\n")
        return results
    
    except Exception as e:
        print(f"❌ Error performing search: {e}")
        raise

# # Example Searches
# query_texts = "get all employees"

schema_result  = schema_search(user_query)

for query in schema_result:
    print(f"🔍 Searching for: {query}")
    #schema_search(query)

In [ ]:
index_name = "query_index"
def query_search(query_text, top_k=3):
    try:
        search_client = SearchClient(
            endpoint=endpoint,
            index_name=index_name,
            credential=credential
        )
        
        results = search_client.search(query_text, top=top_k)
        return  results #[(result["content"], result, result["@search.score"]) for result in results]
    except Exception as e:
        print(f"Error performing search: {e}")
        raise

# Example usage
query_text = "get all employees"
query_results = query_search(query_text)

for result in query_results:
    #text, metadata, score = result
    print(f"id: {result["id"]}")
    print(f"query_desc: {result["query_desc"]}")
    print(f"sql_query: {result["sql_query"]}")
    print(f"Score: {result["@search.score"]}\n")


In [ ]:
# user_query ="get all employees"

In [ ]:
system_prompt = """
You are an expert SQL query generator. Your task is to generate accurate SQL queries based on:  
- A list of predefined queries  
- Schema metadata  

Provided Data:  
Predefined Queries: {predefined_queries}  
Schema Metadata: {schema_result}  

Generate a valid SQL query strictly based on the predefined queries and schema:
"""


In [ ]:
system_prompt

In [ ]:
!pip install groq

In [ ]:
from groq import Groq

client = Groq(api_key="")
completion = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[

       {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_query}
    ],
    temperature=1,
    max_completion_tokens=1024,
    top_p=1,
    stream=True,
    stop=None,
)

for chunk in completion:
    print(chunk.choices[0].delta.content or "", end="")
